In [1]:
%%capture
!pip install transformers
!pip install trl==0.11.3
!pip install datasets
!pip install tqdm
!pip install openpyxl --upgrade
!pip install evaluate
!pip install peft

## Load data

In [1]:
import pandas as pd
from tqdm import tqdm
from datasets import Dataset,concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from trl import SFTTrainer
def split_data(d):
    full_dataset = d.train_test_split(test_size=0.2, shuffle=True, seed=0)
    return full_dataset['train'], full_dataset['test']


df1=pd.read_csv('../ground_truth_series_description.csv')
df1['index_col'] = df1.index
d1 = Dataset.from_pandas(df1)
d1_train,d1_test=split_data(d1)


df2=pd.read_csv('../IU-GroundTruth.csv')
d2 = Dataset.from_pandas(df2)

df3=pd.read_csv('../IU-GroundTruth.csv')
df3 = df3.drop(df3[df3.REPORT.str.contains(pat="IMPRESSION")==False].index)
df3['impression']=True
d3 = Dataset.from_pandas(df3)
d3_train,d3_test=split_data(d3)
d2_train=Dataset.from_dict(d3_train.to_dict())
d2_test=Dataset.from_dict(d3_test.to_dict())
d2_train=d2_train.map(lambda example: {"impression": None})
d2_test=d2_test.map(lambda example: {"impression": None})

dataset_train = concatenate_datasets([d1_train, d2_train,d3_train])
print(len(d1_train),len(d2_train),len(d3_train),len(dataset_train))
dataset_test = concatenate_datasets([d1_test, d2_test,d3_test])
print(len(d1_test),len(d2_test),len(d3_test),len(dataset_test))
dataset_train[0]

/home/runai-home/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 732/732 [00:00<00:00, 10149.86 examples/s]

1116 2927 2927 6970
279 732 732 1743


{'series_description': 'COR CISS T2 1MM MPR',
 'ground_truth_classification': 'T2',
 'index_col': 717,
 'ACCID': None,
 'Report ID': None,
 'Age': None,
 'Sex': None,
 'REPORT': None,
 'No Finding': None,
 'Enlarged Cardiom.': None,
 'Cardiomegaly': None,
 'Lung Lesion': None,
 'Lung Opacity': None,
 'Edema': None,
 'Consolidation': None,
 'Pneumonia': None,
 'Atelectasis': None,
 'Pneumothorax': None,
 'Pleural Effusion': None,
 'Pleural Other': None,
 'Fracture': None,
 'Support Devices': None,
 'impression': None,
 '__index_level_0__': None}

In [3]:
from Trainer.preprocessing import preprocess_function_harmonize_series,preprocess_function_response,preprocess_function_impression
def preprocess_function(example):
    output_text=[]
    #print(example)
    for i in range(len(example['series_description'])):
        if example['series_description'][i] is not None:
            output_text.append(preprocess_function_harmonize_series(example,i))
        elif example['impression'][i]:
            output_text.append(preprocess_function_impression(example,i))
        else:
            output_text.append(preprocess_function_response(example,i))

    return output_text
'''
def preprocess_function(example):
    
    if example['series_description'] is not None:
        return preprocess_function_harmonize_series(example,i)
    elif example['impression']:
        return preprocess_function_impression(example,i)
    else:
        return preprocess_function_response(example,i)
'''

"\ndef preprocess_function(example):\n    \n    if example['series_description'] is not None:\n        return preprocess_function_harmonize_series(example,i)\n    elif example['impression']:\n        return preprocess_function_impression(example,i)\n    else:\n        return preprocess_function_response(example,i)\n"

## Training Config

In [5]:
import os
rank=32

batch_size = 4
num_workers = os.cpu_count()
epochs = 100
bf16 = False
fp16 = True
gradient_accumulation_steps = 8
context_length = 1024
learning_rate = 0.0004
model_name = 'facebook/opt-350m'
out_dir = '../models/opt_350m_3_tasks_new_rank'+str(rank)+'_new' ##CHANGE THIS
seed = 42


from peft import LoraConfig

config = LoraConfig(init_lora_weights=False,r=rank)

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m")
from peft import get_peft_model

lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()
lora_model

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:832: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


trainable params: 3,145,728 || all params: 334,342,144 || trainable%: 0.9409


PeftModel(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTSdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=32, bias=False)
                  )
     

In [30]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_fast=False
)

tokenizer.add_eos_token = True
tokenizer.bos_token = '<s>'

training_args = TrainingArguments(
    output_dir=f"{out_dir}/logs",
    eval_strategy='epoch',
    weight_decay=0.01,
    load_best_model_at_end=False,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    bf16=bf16,
    fp16=fp16,
    report_to='tensorboard',
    num_train_epochs=epochs,
    dataloader_num_workers=num_workers,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    lr_scheduler_type='constant',
    seed=seed
)

trainer = SFTTrainer(
    model=model,
    peft_config=config,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    max_seq_length=context_length,
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=preprocess_function
)

Map: 100%|██████████| 1743/1743 [00:04<00:00, 388.64 examples/s]
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [31]:
trainer.model

PeftModel(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTSdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=32, bias=False)
                  )
     

In [32]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
dataloader = trainer.get_train_dataloader()
for i, sample in enumerate(dataloader):
    print(tokenizer.decode(sample['input_ids'][0]))
    print('#'*50)
    if i == 5:
        break

<s>### Instruction:
Report CXR diagnosis.

### Input:
COMPARISON: None.
INDICATION: Intermittent back pain x2-3 days, now right-sided chest pain under ribs.
FINDINGS: The XXXX examination consists of frontal and lateral radiographs of the chest. The cardiomediastinal contours are within normal limits. Pulmonary vascularity is within normal limits. No focal consolidation, pleural effusion, or pneumothorax identified. The visualized osseous structures and upper abdomen are unremarkable.
IMPRESSION: No evidence of acute cardiopulmonary process.

### Response: 
 {'No Finding': 'Yes', 'Enlarged Cardiom.': 'No', 'Cardiomegaly': 'No', 'Lung Lesion': 'No', 'Lung Opacity': 'No', 'Edema': 'No', 'Consolidation': 'No', 'Pneumonia': 'No', 'Atelectasis': 'No', 'Pneumothorax': 'No', 'Pleural Effusion': 'No', 'Pleural Other': 'No', 'Fracture': 'No', 'Support Devices': 'No'}</s>

##################################################
<s>### Instruction:
Report CXR diagnosis.

### Input:
COMPARISON: None.
I

In [33]:
import warnings
warnings.filterwarnings('ignore')
history = trainer.train()
trainer.model.save_pretrained(out_dir+'/final_model')

Epoch,Training Loss,Validation Loss
0,1.310500,No log
1,0.875800,No log
2,0.783600,No log
3,0.726000,No log
4,0.690200,No log
5,0.660600,No log
6,0.635300,No log
7,0.615700,No log
8,0.595000,No log
9,0.579500,No log


## Eval

In [15]:
import torch
from peft import PeftModel
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
#out_dir = '../models/opt_350m_3_tasks_new_rank32' ##CHANGE THIS

new_model=out_dir+'/final_model'
base_model = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m",
    device_map='cuda:0',
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained('facebook/opt-350m')
out_dir

'../models/opt_350m_3_tasks_new_rank32_new'

## Impression

In [28]:
template = '<s>### Instruction:\nGenerate radiology report based on the following text.\n\n### Input:\n{text}\n\n### Response:'
def summarize(i, tokenizer,  m, template,data):
    if not data[i]['impression']:
        return None,None,None
    t=data[i]['REPORT']
    findings=t.split('IMPRESSION')[0][:-2]
    impression='IMPRESSION '+t.split('IMPRESSION')[1]
    prompt = template.format(text=findings)
    
    prompt_tokenized = tokenizer(
        prompt, 
        return_tensors='pt', 
        return_attention_mask=True
    ).to('cuda')
    output_tokenized = m.generate(
        **prompt_tokenized,
        eos_token_id=tokenizer.eos_token_id,
        max_length=len(prompt_tokenized['input_ids'][0])+500,
        repetition_penalty=1.1,
        temperature=0.8,
        top_k=40,
        top_p=0.1,
        do_sample=True,
        num_beams=5
    )
    #print(prompt)
    answer = tokenizer.decode(token_ids=output_tokenized[0][len(prompt_tokenized[0]):]).strip().replace("'", "\"")[:-4]
    #print(impression,answer)
    
    return findings,impression,answer



In [29]:
from tqdm import tqdm
import numpy as np
import evaluate
rouge = evaluate.load('bleu')
eval_set=len(dataset_test)
data_collect=[]
col=[]
for i in tqdm(range(eval_set)):
    inputs,ground_truth,answer=summarize(i, tokenizer, model, template,dataset_test)
    if ground_truth is None:
        continue
    data_collect.append([inputs,ground_truth,answer])
    results = rouge.compute(predictions=[answer], references=[ground_truth])
    col.append(results)
    #print(results)
np.save("data.npy", np.array(data_collect))

d = {}
for k in col[0].keys():
    d[k] = [i[k] for i in col]
#print(d)
d_avg={}
for k in d.keys():
    d_avg[k]=np.mean(d[k])
d_avg

100%|██████████| 1743/1743 [04:33<00:00,  6.37it/s] 


{'bleu': 0.5175948054499104,
 'precisions': 0.6160519720624272,
 'brevity_penalty': 0.8534236821333889,
 'length_ratio': 1.004139827967369,
 'translation_length': 12.092896174863387,
 'reference_length': 13.78825136612022}

In [37]:
from tqdm import tqdm
import numpy as np
import evaluate
rouge = evaluate.load('rouge')
eval_set=len(dataset_test)

col=[]
for i in tqdm(range(eval_set)):
    ground_truth,answer=summarize(i, tokenizer, model, template,dataset_test)
    if ground_truth is None:
        continue
    results = rouge.compute(predictions=[answer], references=[ground_truth])
    col.append(results)
    #print(results)
d = {}
for k in col[0].keys():
    d[k] = [i[k] for i in col]
#print(d)
d_avg={}
for k in d.keys():
    d_avg[k]=np.mean(d[k])
d_avg

100%|██████████| 1743/1743 [06:04<00:00,  4.79it/s] 


{'rouge1': 0.6884594854823676,
 'rouge2': 0.5723254944581285,
 'rougeL': 0.6762677993224715,
 'rougeLsum': 0.6762677993224715}

## Reponse CXR

In [60]:
template = '<s>### Instruction:\nReport diagnosis.\n\n### Input:\n{text}\n\n### Response:'
def summarize(i, tokenizer,  m, template):
    if dataset_test[i]['impression'] or dataset_test[i]['series_description']:
        return None,None
    text = dataset_test[i]['REPORT']
    accid=dataset_test[i]['ACCID']
    with open("../response/"+str(accid)+".json") as f:
            ground_truth = json.load(f)
    prompt = template.format(text=text)
    prompt_tokenized = tokenizer(
        prompt, 
        return_tensors='pt', 
        return_attention_mask=True
    ).to('cuda')
    output_tokenized = m.generate(
        **prompt_tokenized,
        eos_token_id=tokenizer.eos_token_id,
        max_length=len(prompt_tokenized['input_ids'][0])+500,
        repetition_penalty=1.1,
        temperature=0.8,
        top_k=40,
        top_p=0.1,
        do_sample=True,
        num_beams=5
    )
    #print(prompt)
    answer = tokenizer.decode(token_ids=output_tokenized[0][len(prompt_tokenized[0]):]).strip().replace("'", "\"").split('}')[0]+'}'
    #print(answer)
    try:
        answer_dict=json.loads(answer)
    except:
        print(answer)
        return ground_truth,dict()
    return ground_truth,answer_dict

In [61]:
accid=dataset_test[500]['ACCID']
with open("../response/"+str(accid)+".json") as f:
    ground_truth = json.load(f)
count_collection={}
for i in ground_truth.keys():
    count_collection[i]=[]
    
for i in tqdm(range(eval_set)):
    if dataset_test[i]['impression'] or dataset_test[i]['series_description']:
        continue
    text = dataset_test[i]['REPORT']
    accid=dataset_test[i]['ACCID']
    with open("../response/"+str(accid)+".json") as f:
            ground_truth = json.load(f)
    for i in ground_truth.keys():
        count_collection[i].append(ground_truth[i])

100%|██████████| 1743/1743 [00:01<00:00, 1086.11it/s]


In [62]:
from tqdm import tqdm
import numpy as np
import json
c=0

accid=dataset_test[500]['ACCID']
with open("../response/"+str(accid)+".json") as f:
    ground_truth = json.load(f)
acc_collection={}
for i in ground_truth.keys():
    acc_collection[i]=[]


eval_set=len(dataset_test)
for i in tqdm(range(eval_set)):
    ground_truth,answer_dict=summarize(i, tokenizer, model, template)
    if ground_truth is None:
        continue
    for i in ground_truth.keys():
        true= 1 if ground_truth[i]=='Yes' else 0
        pred= 1 if answer_dict[i]=='Yes' else 0
        acc_collection[i].append([true,pred])

100%|██████████| 1743/1743 [21:28<00:00,  1.35it/s] 


In [64]:
for i in acc_collection.keys():
    a=np.array(acc_collection[i])
    if sum(a[:,1])==0:
        print('bad')
    print("\t".join([str(np.round(accuracy_score(a[:,0],a[:,1]),5)),str(np.round(f1_score(a[:,0],a[:,1]),5))]))

0.97268	0.96466
0.96585	0.83444
0.98497	0.91971
0.96721	0.9469
0.96858	0.94831
0.9959	0.88889
0.99863	0.90909
0.99727	0.8
0.98634	0.91379
1.0	1.0
1.0	1.0
0.99317	0.70588
0.99454	0.81818
0.97951	0.86239


In [65]:
from sklearn.metrics import f1_score,accuracy_score

for i in acc_collection.keys():
    a=np.array(acc_collection[i])
    if sum(a[:,1])==0:
        print('bad')
    print(np.round(accuracy_score(a[:,0],a[:,1]),3),np.round(f1_score(a[:,0],a[:,1]),3))

0.973 0.965
0.966 0.834
0.985 0.92
0.967 0.947
0.969 0.948
0.996 0.889
0.999 0.909
0.997 0.8
0.986 0.914
1.0 1.0
1.0 1.0
0.993 0.706
0.995 0.818
0.98 0.862


## Series Harmonization

In [49]:
template = "<s>### Instruction:\n{instruction}\n\n### Input:\n{text}\n\n### Response:"
instruction = 'Harmonize sequence name.'
import json
def summarize(i, tokenizer,  m, template):
    if dataset_test[i]['series_description'] is None:
        return None,None
    text = dataset_test[i]['series_description']
    idx=dataset_test[i]['index_col']
    with open("../harmonize_series/"+str(idx)+".json") as f:
            ground_truth = json.load(f)
            
    prompt = template.format(instruction=instruction, text=text)
    prompt_tokenized = tokenizer(
        prompt, 
        return_tensors='pt', 
        return_attention_mask=True
    ).to('cuda')
    output_tokenized = m.generate(
        **prompt_tokenized,
        eos_token_id=tokenizer.eos_token_id,
        max_length=len(prompt_tokenized['input_ids'][0])+500,
        repetition_penalty=1.1,
        temperature=0.8,
        top_k=40,
        top_p=0.1,
        do_sample=True,
        num_beams=5
    )
    
    answer = tokenizer.decode(token_ids=output_tokenized[0][len(prompt_tokenized[0]):]).strip().replace("'", "\"").split('}')[0]+'}'
    try:
        answer_dict=json.loads(answer)
    except:
        print(answer)
        return ground_truth,dict()
    return ground_truth,answer_dict

In [50]:
!pip install tqdm
from tqdm import tqdm
import numpy as np
c=0
acc=[]

eval_set=len(dataset_test)
for i in tqdm(range(eval_set)):
    ground_truth,answer_dict=summarize(i, tokenizer, model, template)
    if ground_truth is None:
        continue
    if ground_truth==answer_dict:
        acc.append(1)
    else:
        acc.append(0)
            
np.mean(acc)

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip


100%|██████████| 1743/1743 [03:05<00:00,  9.38it/s] 


0.974910394265233